### LIBRARY IMPORTS

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy
import warnings

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
from sklearn.model_selection import StratifiedKFold

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor
from src.gb_classifier import GBClassifier

warnings.simplefilter(action='ignore', category=FutureWarning)

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]
target = active_dataset_config["target"]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

raw_train, raw_test = data_manager.load_raw_data()
raw_train = processor.cut_data(raw_train)

### STRATIFIED CROSS-VALIDATION

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def my_cross_validation(model, data, target, skf, processor):
    log_loss_scores = []
    accuracy_scores = []
    oof_preds = np.zeros(len(data))

    for train_idx, valid_idx in skf.split(data, data[target]):
        train_df = raw_train.iloc[train_idx]
        valid_df = raw_train.iloc[valid_idx]

        train_proc = processor.fit_transform(train_df)
        valid_proc = processor.transform(valid_df)

        X_train, y_train = train_proc.drop(columns=[target]), train_proc[target]
        X_valid, y_valid = valid_proc.drop(columns=[target]), valid_proc[target]

        model.fit(X_train, y_train)

        preds = model.predict(X_valid)
        oof_preds[valid_idx] = preds

        fold_log_loss = log_loss(y_valid, preds)
        fold_accuracy = accuracy_score(y_valid, preds)
        
        log_loss_scores.append(fold_log_loss)
        accuracy_scores.append(fold_accuracy)
    
    return log_loss_scores, accuracy_scores, oof_preds

### LOGISTIC REGRESSION

In [4]:
%%time

logistic = LogisticRegression()
logistic_log_loss_scores, logistic_accuracy_scores, logistic_oof_preds = my_cross_validation(logistic, raw_train, target, skf, processor)

CPU times: total: 3.06 s
Wall time: 2.15 s


In [5]:
print("Logistic regression log loss:", logistic_log_loss_scores)
print("Logistic regression accuracy:", logistic_accuracy_scores)

Logistic regression log loss: [4.015262987547651, 3.8981211140330196, 4.110778669028812, 4.063921919622959, 4.247744551907457]
Logistic regression accuracy: [0.8886, 0.89185, 0.88595, 0.88725, 0.88215]


### DECISION TREE

In [15]:
%%time

dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt_log_loss_scores, dt_accuracy_scores, dt_oof_preds = my_cross_validation(dt, raw_train, target, skf, processor)

CPU times: total: 2.38 s
Wall time: 2.37 s


In [16]:
print("Decision tree log loss:", dt_log_loss_scores)
print("Decision tree accuracy:", dt_accuracy_scores)

Decision tree log loss: [4.483830481606174, 4.327040589363515, 4.471215202919983, 4.3702929734304545, 4.564928701731688]
Decision tree accuracy: [0.8756, 0.87995, 0.87595, 0.87875, 0.87335]


### RANDOM FOREST

In [19]:
%%time

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_log_loss_scores, rf_accuracy_scores, rf_oof_preds = my_cross_validation(rf, raw_train, target, skf, processor)

CPU times: total: 17.3 s
Wall time: 17.4 s


In [20]:
print("Random forest log loss:", rf_log_loss_scores)
print("Random forest accuracy:", rf_accuracy_scores)

Random forest log loss: [4.226118359873986, 4.080141563648062, 4.26756856127147, 4.26396419593256, 4.4766217509283495]
Random forest accuracy: [0.88275, 0.8868, 0.8816, 0.8817, 0.8758]


### NEURAL NETWORK

In [26]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        patience: int = 10
    ) -> None:
        input_size = X_train.shape[1]
        
        unique_classes = np.unique(y_train)
        output_size = len(unique_classes)
        
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.long).to(self.device).squeeze()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.to(torch.long).to(self.device).squeeze())
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            print(f"Epoch: {epoch + 1} | Validation log loss: {val_loss:.4f} | Validation accuracy {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        X_t = torch.from_numpy(X).to(torch.float32).to(self.device)
        self.eval()
        with torch.no_grad():
            logits = self.forward(X_t)
            predictions = torch.argmax(logits, dim=1)
        
        return predictions.cpu().numpy()

In [ ]:
%%time

mlp = MLP(epochs=100, learning_rate=0.0005, hidden_size=[16, 8], batch_size=256)

mlp_log_loss_scores = []
mlp_accuracy_scores = []
mlp_oof_preds = np.zeros(len(raw_train))

for train_idx, valid_idx in skf.split(raw_train, raw_train[target]):
    train_df = raw_train.iloc[train_idx]
    valid_df = raw_train.iloc[valid_idx]

    train_proc = processor.fit_transform(train_df)
    valid_proc = processor.transform(valid_df)

    X_train, y_train = train_proc.drop(columns=[target]), train_proc[target]
    X_valid, y_valid = valid_proc.drop(columns=[target]), valid_proc[target]

    mlp.fit(X_train.values, y_train.values, X_valid.values, y_valid.values)

    preds = mlp.predict(X_valid.values).ravel()
    mlp_oof_preds[valid_idx] = preds

    fold_log_loss = log_loss(y_valid, preds)
    fold_accuracy = accuracy_score(y_valid, preds)
        
    mlp_log_loss_scores.append(fold_log_loss)
    mlp_accuracy_scores.append(fold_accuracy)

    print('-' * 50)

Epoch: 1 | Validation log loss: 0.3326 | Validation accuracy 0.8764
Epoch: 2 | Validation log loss: 0.2913 | Validation accuracy 0.8806
Epoch: 3 | Validation log loss: 0.2820 | Validation accuracy 0.8835
Epoch: 4 | Validation log loss: 0.2756 | Validation accuracy 0.8862
Epoch: 5 | Validation log loss: 0.2734 | Validation accuracy 0.8877
Epoch: 6 | Validation log loss: 0.2712 | Validation accuracy 0.8878
Epoch: 7 | Validation log loss: 0.2728 | Validation accuracy 0.8866
Epoch: 8 | Validation log loss: 0.2699 | Validation accuracy 0.8888
Epoch: 9 | Validation log loss: 0.2698 | Validation accuracy 0.8893
Epoch: 10 | Validation log loss: 0.2701 | Validation accuracy 0.8885
Epoch: 11 | Validation log loss: 0.2699 | Validation accuracy 0.8890
Epoch: 12 | Validation log loss: 0.2694 | Validation accuracy 0.8887
Epoch: 13 | Validation log loss: 0.2694 | Validation accuracy 0.8891
Epoch: 14 | Validation log loss: 0.2694 | Validation accuracy 0.8888
Epoch: 15 | Validation log loss: 0.2698 | V

KeyboardInterrupt: 

In [25]:
rf_oof_preds

array([0., 1., 0., ..., 0., 1., 1.], shape=(100000,))